In [1]:
import pandas as pd
import numpy as np
import os

updated_path = r"D:\Swapnil\Work\Projects\FIFA\Updated Base Files"

df = pd.read_csv(os.path.join(updated_path, "Step7_Global_With_Customers.csv"))

print(df.shape)

(1389312, 47)


In [2]:
campaign_base = df[["company_name", "country"]].drop_duplicates().copy()

print(campaign_base.shape)  # should be ~144 (48 countries × 3 companies)

(144, 2)


In [3]:
np.random.seed(42)

campaign_base["campaign_spend"] = np.random.randint(5000, 50000, len(campaign_base))

In [4]:
campaign_base["impressions"] = campaign_base["campaign_spend"] * np.random.randint(40, 120, len(campaign_base))

In [5]:
campaign_base["clicks"] = (
    campaign_base["impressions"] *
    np.random.uniform(0.015, 0.07, len(campaign_base))
).astype(int)

In [6]:
campaign_base["conversions"] = (
    campaign_base["clicks"] *
    np.random.uniform(0.03, 0.18, len(campaign_base))
).astype(int)

In [7]:
campaign_base["discount_percent"] = np.random.choice(
    [0, 5, 10, 15, 20],
    size=len(campaign_base),
    p=[0.15, 0.25, 0.30, 0.20, 0.10]
)

In [8]:
campaign_base["discount_percent"] = np.random.choice(
    [0, 5, 10, 15, 20],
    size=len(campaign_base),
    p=[0.15, 0.25, 0.30, 0.20, 0.10]
)

In [9]:
discount_mask = df["discount_percent"] >= 10

df.loc[discount_mask, "fifa_units_sold"] *= 1.10
df.loc[discount_mask, "fifa_total_sales"] *= 1.08

KeyError: 'discount_percent'

In [10]:
print(df.columns)

Index(['sales_id', 'date', 'retailer', 'retailer_id', 'us_region', 'state',
       'city', 'product', 'price_per_unit', 'units_sold', 'total_sales',
       'operating_profit', 'operating_margin', 'country_id', 'country',
       'region', 'confederation', 'is_world_cup_team', 'is_host', 'tier',
       'country_multiplier', 'company_name', 'sim_units_sold',
       'sim_price_per_unit', 'sim_total_sales', 'sim_operating_profit',
       'match_day_flag', 'match_count', 'fifa_total_sales', 'fifa_units_sold',
       'fifa_operating_profit', 'knockout_stage_flag', 'fifa_sales_uplift_pct',
       'customer_id', 'age', 'gender', 'category', 'purchase_amount',
       'location', 'season', 'review_rating', 'discount_applied',
       'promo_code_used', 'previous_purchases', 'payment_method',
       'purchase_frequency', 'age_group'],
      dtype='object')


In [11]:
# Step 8 recovery: create + merge campaign data

campaign_base = df[["company_name", "country"]].drop_duplicates().copy()

np.random.seed(42)

campaign_base["campaign_spend"] = np.random.randint(5000, 50000, len(campaign_base))

campaign_base["impressions"] = (
    campaign_base["campaign_spend"] *
    np.random.randint(40, 120, len(campaign_base))
)

campaign_base["clicks"] = (
    campaign_base["impressions"] *
    np.random.uniform(0.015, 0.07, len(campaign_base))
).astype(int)

campaign_base["conversions"] = (
    campaign_base["clicks"] *
    np.random.uniform(0.03, 0.18, len(campaign_base))
).astype(int)

campaign_base["discount_percent"] = np.random.choice(
    [0, 5, 10, 15, 20],
    size=len(campaign_base),
    p=[0.15, 0.25, 0.30, 0.20, 0.10]
)

df = df.merge(
    campaign_base,
    on=["company_name", "country"],
    how="left"
)

print("discount_percent exists:", "discount_percent" in df.columns)
print(df[["company_name", "country", "campaign_spend", "discount_percent"]].head())

discount_percent exists: True
  company_name        country  campaign_spend  discount_percent
0    Coca-Cola         Canada           20795                 5
1    Coca-Cola         Mexico            5860                15
2    Coca-Cola  United States           43158                 5
3    Coca-Cola      Argentina           49732                15
4    Coca-Cola         Brazil           16284                10


In [12]:
discount_mask = df["discount_percent"] >= 10

df.loc[discount_mask, "fifa_units_sold"] *= 1.10
df.loc[discount_mask, "fifa_total_sales"] *= 1.08

df["ctr"] = df["clicks"] / df["impressions"]
df["conversion_rate"] = df["conversions"] / df["clicks"]
df["roas"] = df["fifa_total_sales"] / df["campaign_spend"]

df[["ctr", "conversion_rate", "roas"]] = df[["ctr", "conversion_rate", "roas"]].replace(
    [np.inf, -np.inf], 0
).fillna(0)

output_path = os.path.join(updated_path, "Step8_Final_Dataset.csv")
df.to_csv(output_path, index=False)

print("Step 8 saved:", output_path)

Step 8 saved: D:\Swapnil\Work\Projects\FIFA\Updated Base Files\Step8_Final_Dataset.csv
